In [2]:
# Part 1: Data Loading and Exploration
import pandas as pd
import numpy as np


from sklearn.datasets import fetch_california_housing 

#Loading the dataset from sklearn
housing = fetch_california_housing()

#Creating a Pandas DataFrame for the features and a Series for the target variable
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name="med_house_value")

print("X shape:", X.shape)
print("y shape:", y.shape)

#displaying the first five rows of the dataset
print("First 5 rows of X:")
print(X.head())
print("\nFirst 5 values of y:")
print(y.head())

# printing feature names and checking for missing values
print("Feature names:")
print(list(X.columns))
print("\nMissing values per column:")
print(X.isna().sum())
print("\nMissing values in target:")
print(y.isna().sum())

#generate summary statstics 
print("Summary statistics for features:")
print(X.describe())

print("\nSummary statistics for target:")
print(y.describe())

X shape: (20640, 8)
y shape: (20640,)
First 5 rows of X:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  

First 5 values of y:
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: med_house_value, dtype: float64
Feature names:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Missing values per column:
MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int6

In [3]:
# Part 2: Linear Regression on Unscaled Data

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Split the dataset into training and test sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Train/Test Split:")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Train a linear regression model on the unscaled data
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

print("\nModel trained on unscaled data.")

# Make predictions on the test set
y_pred = lin_reg.predict(X_test)

print("\nFirst 10 predictions vs actual values:")
preview_df = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
})
print(preview_df)

# Evaluate model performance
# Mean Squared Error (MSE)
mse = np.mean((y_test - y_pred) ** 2)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# R^2 Score (manual calculation)
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print("\nModel Performance Metrics:")
print("MSE:", mse)
print("RMSE:", rmse)
print("R^2:", r2)

# Check which features have the strongest impact using coefficients
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lin_reg.coef_
})

coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

print("\nModel Coefficients (sorted by absolute value):")
print(coef_df.sort_values("Abs_Coefficient", ascending=False))

Train/Test Split:
X_train shape: (16512, 8)
X_test shape: (4128, 8)
y_train shape: (16512,)
y_test shape: (4128,)

Model trained on unscaled data.

First 10 predictions vs actual values:
    Actual  Predicted
0  0.47700   0.719123
1  0.45800   1.764017
2  5.00001   2.709659
3  2.18600   2.838926
4  2.78000   2.604657
5  1.58700   2.011754
6  1.98200   2.645500
7  1.57500   2.168755
8  3.40000   2.740746
9  4.46600   3.915615

Model Performance Metrics:
MSE: 0.5558915986952442
RMSE: 0.7455813830127763
R^2: 0.575787706032451

Model Coefficients (sorted by absolute value):
      Feature  Coefficient  Abs_Coefficient
3   AveBedrms     0.783145         0.783145
0      MedInc     0.448675         0.448675
7   Longitude    -0.433708         0.433708
6    Latitude    -0.419792         0.419792
2    AveRooms    -0.123323         0.123323
1    HouseAge     0.009724         0.009724
5    AveOccup    -0.003526         0.003526
4  Population    -0.000002         0.000002


1. What does the R² score tell us about model performance? 
    R² score shows us how  how much variation is explained by the model for the median house values. The more the R² is, the better the model is at predicting the data. For example, if the R² is around 0.60, this means the model is doing a good job at explaining 60% of the variation in the house values.

2. Which features seem to have the strongest impact on predictions based on the model’s coefficients?
    The feature with the greatest absolute coefficient value has the greatest impact on the model's prediction. A positive coefficient value indicates that if a feature increases, then the house value will also increase, while a negative coefficient value indicates that if a feature increases, then the house value will actually decrease. I reviewed the coefficient table sorted by absolute value to determine which features the model uses most heavily.

3. How well do the predicted values match the actual values?
    The predicted values are generally within the same range as the actual values. However, the values are not exact. The root-mean-square error (RMSE) is a clear measure of the error in the prediction values because it is given in the same units as the target variable. When the RMSE is relatively low and the coefficient of determination (R²) is relatively high, the values align with the actual values fairly well, yet there is still some error present because there are many complex factors involved with the prices of houses.

In [4]:
# Part 3: Feature Selection and Simplified Model

# Selecting three features
selected_features = ["MedInc", "Population", "Longitude"]

X_simplified = X[selected_features]

print("Selected Features:")
print(selected_features)

# Split the simplified dataset
X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(
    X_simplified, y, test_size=0.20, random_state=42
)

print("\nSimplified Train/Test Split:")
print("X_train shape:", X_train_simple.shape)
print("X_test shape:", X_test_simple.shape)

# Train the simplified linear regression model
lin_reg_simple = LinearRegression()
lin_reg_simple.fit(X_train_simple, y_train_simple)

print("\nSimplified model trained.")

# Make predictions
y_pred_simple = lin_reg_simple.predict(X_test_simple)

# Evaluate performance
mse_simple = np.mean((y_test_simple - y_pred_simple) ** 2)
rmse_simple = np.sqrt(mse_simple)

ss_res_simple = np.sum((y_test_simple - y_pred_simple) ** 2)
ss_tot_simple = np.sum((y_test_simple - np.mean(y_test_simple)) ** 2)
r2_simple = 1 - (ss_res_simple / ss_tot_simple)

print("\nSimplified Model Performance:")
print("MSE:", mse_simple)
print("RMSE:", rmse_simple)
print("R^2:", r2_simple)

# Compare to full model
print("\nComparison with Full Model:")
print("Full Model R^2:", r2)
print("Simplified Model R^2:", r2_simple)
print("Full Model RMSE:", rmse)
print("Simplified Model RMSE:", rmse_simple)

# Comparison table
compare_df = pd.DataFrame({
    "Model": ["Full Model", "Simplified Model (3 features)"],
    "MSE": [mse, mse_simple],
    "RMSE": [rmse, rmse_simple],
    "R2": [r2, r2_simple]
})

print("\nFull vs Simplified Comparison:")
print(compare_df)


Selected Features:
['MedInc', 'Population', 'Longitude']

Simplified Train/Test Split:
X_train shape: (16512, 3)
X_test shape: (4128, 3)

Simplified model trained.

Simplified Model Performance:
MSE: 0.70664685548632
RMSE: 0.8406228973126535
R^2: 0.46074327387856673

Comparison with Full Model:
Full Model R^2: 0.575787706032451
Simplified Model R^2: 0.46074327387856673
Full Model RMSE: 0.7455813830127763
Simplified Model RMSE: 0.8406228973126535

Full vs Simplified Comparison:
                           Model       MSE      RMSE        R2
0                     Full Model  0.555892  0.745581  0.575788
1  Simplified Model (3 features)  0.706647  0.840623  0.460743


1.  How does the simplified model compare to the full model?
    The simplified model tends to perform at a lower level compared to the full model. For the full model, the R² value is around 0.576. This indicates that the model explains 57.6% of the total variation in the housing prices. On the other hand, the simplified model has a lower R² value of around 0.461. This suggests that the model explains 46.1% of the total variation in the housing prices. Additionally, the simplified model has a higher RMSE compared to the full model. This suggests that the simplified model has larger prediction errors compared to the full model. The RMSE values are 0.841 and 0.746 for the simplified and full models, respectively.

2.  Would you use this simplified model in practice? Why or why not?
    I would use the simplified model only if interpretability or simplicity is considered more important than the accuracy of the predictions. Although the simplified model uses fewer features and is more interpretable, it also accounts for 11% less of the variance in house prices compared to the full model, and the prediction errors are larger. Therefore, if the primary concern is the accuracy of the predictions, the full model should be considered more appropriate. However, if the primary concern is the simplicity of the computation or the explanation of the model, the simplified model could be useful.

